# Week 3 — CNN Fundamentals
## Same Data, Smarter Architecture

**IIT Madras · Wadhwani School of AI**

---

**Session Plan (60 min):**
1. Convolution Intuition (~10 min) — what kernels actually do on real images
2. Build & Train CNN (~17 min) — Fashion-MNIST with conv layers, compare to week 1's MLP
3. Visualize What the CNN Learned (~10 min) — filters, feature maps, the interpretability payoff
4. Q&A (~10 min)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")

---
# Part 1: Convolution Intuition (~10 min)

Before building a CNN, let's see what convolution actually does. We'll apply hand-crafted kernels to a real image and watch the output.

### 1.1 Load a Sample Image

We'll grab one Fashion-MNIST image and use it as our test subject.

In [ ]:
# Load Fashion-MNIST (we'll use it throughout)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Grab a sample image (unnormalized for display)
raw_transform = transforms.ToTensor()
raw_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=raw_transform
)

sample_img, sample_label = raw_dataset[0]
print(f"Image shape: {sample_img.shape}  (C×H×W)")
print(f"Label: {class_names[sample_label]}")

plt.figure(figsize=(3, 3))
plt.imshow(sample_img.squeeze(), cmap='gray')
plt.title(f'{class_names[sample_label]}', fontweight='bold')
plt.axis('off')
plt.show()

### 1.2 Hand-Crafted Kernels

These kernels are designed by hand to detect specific patterns. A CNN *learns* kernels like these automatically — but seeing them manually first builds intuition.

In [ ]:
# Define some classic kernels
kernels = {
    'Vertical Edges': torch.tensor([[-1, 0, 1],
                                     [-1, 0, 1],
                                     [-1, 0, 1]], dtype=torch.float32),

    'Horizontal Edges': torch.tensor([[-1, -1, -1],
                                       [ 0,  0,  0],
                                       [ 1,  1,  1]], dtype=torch.float32),

    'Sharpen': torch.tensor([[ 0, -1,  0],
                              [-1,  5, -1],
                              [ 0, -1,  0]], dtype=torch.float32),

    'Blur (Average)': torch.ones(3, 3, dtype=torch.float32) / 9.0,
}

# Apply each kernel using F.conv2d
fig, axes = plt.subplots(2, len(kernels) + 1, figsize=(16, 7))

# Row 1: kernel visualization
axes[0, 0].imshow(sample_img.squeeze(), cmap='gray')
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')
axes[1, 0].axis('off')

for idx, (name, kernel) in enumerate(kernels.items(), 1):
    # Show the kernel
    axes[0, idx].imshow(kernel.numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
    axes[0, idx].set_title(f'Kernel: {name}', fontsize=10)
    for (r, c), val in np.ndenumerate(kernel.numpy()):
        axes[0, idx].text(c, r, f'{val:.1f}', ha='center', va='center',
                          fontsize=9, color='black', fontweight='bold')
    axes[0, idx].axis('off')

    # Apply convolution using F.conv2d
    # Input: (batch, channels, H, W), Kernel: (out_ch, in_ch, kH, kW)
    kernel_4d = kernel.unsqueeze(0).unsqueeze(0)  # 1×1×3×3
    output = F.conv2d(sample_img.unsqueeze(0), kernel_4d, padding=1)

    axes[1, idx].imshow(output.squeeze().detach().numpy(), cmap='gray')
    axes[1, idx].set_title(f'After {name}', fontsize=10, fontweight='bold')
    axes[1, idx].axis('off')

fig.suptitle('Convolution in Action — Same Image, Different Kernels', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Each kernel detects a different pattern.")
print("A CNN LEARNS these kernels automatically during training.")

### 1.3 What nn.Conv2d Does Under the Hood

Quick look at the key parameters. This is all you need to know to use conv layers.

In [ ]:
# One convolutional layer — what happens to the shape?
conv_layer = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)

# Pass our sample image through
x = sample_img.unsqueeze(0)  # Add batch dim: 1×1×28×28
out = conv_layer(x)

print(f"Input shape:  {x.shape}      → (batch, channels, H, W)")
print(f"Output shape: {out.shape}   → (batch, 32 filters, H, W)")
print(f"")
print(f"Parameters in this layer:")
print(f"  Weights: {conv_layer.weight.shape}  → 32 filters × 1 channel × 3×3 kernel")
print(f"  Bias:    {conv_layer.bias.shape}")
print(f"  Total:   {sum(p.numel() for p in conv_layer.parameters()):,} params")
print(f"")
print(f"Compare to MLP first layer: 784 × 128 = {784*128:,} params")
print(f"That's {784*128 // sum(p.numel() for p in conv_layer.parameters())}× more!")

### 1.4 Pooling — Shrink and Keep the Best

After convolution, MaxPool2d halves the spatial dimensions by keeping the maximum value in each 2×2 window.

In [ ]:
pool = nn.MaxPool2d(kernel_size=2, stride=2)

pooled = pool(out)

print(f"Before pooling: {out.shape}")
print(f"After pooling:  {pooled.shape}")
print(f"")
print(f"Spatial dimensions: 28×28 → 14×14 (halved)")
print(f"Channels: still 32 (pooling doesn't change channel count)")
print(f"")
print(f"Size walkthrough for our full CNN:")
print(f"  1×28×28 → Conv(32) → Pool → 32×14×14")
print(f"         → Conv(64) → Pool → 64×7×7")
print(f"         → Conv(128) → Pool → 128×3×3")
print(f"         → Flatten → 1,152 → MLP classifier → 10 classes")

---
# Part 2: Build & Train CNN (~17 min)

Now we build a real CNN and train it on Fashion-MNIST — the same dataset we used with an MLP in week 1. Same training loop, different architecture.

### 2.1 The MLP Baseline (Week 1 Recap)

First, let's quickly re-train the MLP from week 1 so we have a fair comparison. Same data, same epochs, same optimizer.

In [ ]:
# Week 1's MLP architecture (for comparison)
class FashionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),              # 1×28×28 → 784
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.net(x)

mlp_params = sum(p.numel() for p in FashionMLP().parameters())
print(f"MLP parameters: {mlp_params:,}")
print(f"First layer alone: 784 × 128 + 128 = {784*128+128:,}")

### 2.2 CNN Architecture

Three conv blocks (Conv → ReLU → Pool), then flatten into a small classifier. The new layer types are `nn.Conv2d`, `nn.MaxPool2d`, and `nn.BatchNorm2d`.

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # Conv block 1: 1×28×28 → 32×14×14
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)

        # Conv block 2: 32×14×14 → 64×7×7
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Conv block 3: 64×7×7 → 128×3×3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # Classifier head (same idea as MLP!)
        self.classifier = nn.Sequential(
            nn.Flatten(),              # 128×3×3 = 1152
            nn.Linear(128 * 3 * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)         # 10 classes
        )

    def forward(self, x):
        # Block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # → 32×14×14
        # Block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # → 64×7×7
        # Block 3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # → 128×3×3
        # Classify
        x = self.classifier(x)
        return x

cnn = FashionCNN()
cnn_params = sum(p.numel() for p in cnn.parameters())

print(f"CNN parameters: {cnn_params:,}")
print(f"MLP parameters: {mlp_params:,}")
print(f"")
print(f"CNN has more params here, but the conv layers are much more")
print(f"efficient per parameter — each one covers the ENTIRE image.")

# Verify shapes
test_input = torch.randn(1, 1, 28, 28)
test_output = cnn(test_input)
print(f"\nShape check: {test_input.shape} → {test_output.shape}")

### 2.3 Train Both Models

Same training function, same hyperparameters. The only difference is the model architecture. This is the fairest comparison possible.

In [ ]:
# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Unified training function — same 5-step loop for both models
def train_and_evaluate(model, train_loader, test_loader, num_epochs=10, lr=0.001):
    model = model.to(device)
    loss_fn = nn.CrossEntropyLoss()           # Same loss
    optimizer = optim.Adam(model.parameters(), lr=lr)  # Same optimizer

    train_losses, test_losses, test_accs = [], [], []
    start = time.time()

    for epoch in range(num_epochs):
        # ===== TRAIN =====
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            pred = model(X_batch)           # 1. Forward
            loss = loss_fn(pred, y_batch)    # 2. Loss
            optimizer.zero_grad()            # 3. Zero grad
            loss.backward()                  # 4. Backward
            optimizer.step()                 # 5. Update

            running_loss += loss.item()

        train_losses.append(running_loss / len(train_loader))

        # ===== TEST =====
        model.eval()
        test_loss, correct, total = 0.0, 0, 0
        with torch.inference_mode():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                pred = model(X_batch)
                test_loss += loss_fn(pred, y_batch).item()
                correct += (pred.argmax(1) == y_batch).sum().item()
                total += y_batch.size(0)

        test_losses.append(test_loss / len(test_loader))
        test_accs.append(correct / total)

        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:2d}/{num_epochs} | "
                  f"Train Loss: {train_losses[-1]:.4f} | "
                  f"Test Acc: {test_accs[-1]:.1%}")

    elapsed = time.time() - start
    print(f"  Done in {elapsed:.1f}s | Final accuracy: {test_accs[-1]:.1%}\n")
    return train_losses, test_losses, test_accs

In [ ]:
NUM_EPOCHS = 10

# Train MLP
print("=" * 50)
print("Training MLP (week 1 architecture)")
print("=" * 50)
mlp_model = FashionMLP()
mlp_train_loss, mlp_test_loss, mlp_test_acc = train_and_evaluate(
    mlp_model, train_loader, test_loader, num_epochs=NUM_EPOCHS
)

# Train CNN
print("=" * 50)
print("Training CNN (today's architecture)")
print("=" * 50)
cnn_model = FashionCNN()
cnn_train_loss, cnn_test_loss, cnn_test_acc = train_and_evaluate(
    cnn_model, train_loader, test_loader, num_epochs=NUM_EPOCHS
)

### 2.4 Head-to-Head Comparison

Same dataset. Same training loop. Same number of epochs. The only difference is the architecture.

In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Train loss
axes[0].plot(epochs_range, mlp_train_loss, 'o-', label='MLP', color='#EF4444', linewidth=2)
axes[0].plot(epochs_range, cnn_train_loss, 's-', label='CNN', color='#4ADE80', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test loss
axes[1].plot(epochs_range, mlp_test_loss, 'o-', label='MLP', color='#EF4444', linewidth=2)
axes[1].plot(epochs_range, cnn_test_loss, 's-', label='CNN', color='#4ADE80', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Test Loss', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Test accuracy
axes[2].plot(epochs_range, [a*100 for a in mlp_test_acc], 'o-', label='MLP', color='#EF4444', linewidth=2)
axes[2].plot(epochs_range, [a*100 for a in cnn_test_acc], 's-', label='CNN', color='#4ADE80', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_title('Test Accuracy', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

fig.suptitle('MLP vs CNN — Same Data, Same Training Loop', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nFinal Accuracy:")
print(f"  MLP: {mlp_test_acc[-1]:.1%}")
print(f"  CNN: {cnn_test_acc[-1]:.1%}")
print(f"  Improvement: +{(cnn_test_acc[-1] - mlp_test_acc[-1])*100:.1f} percentage points")
print(f"\nSame training loop. Architecture matters.")

### 2.5 Per-Class Comparison

Where exactly does the CNN improve over the MLP? Some classes benefit more from spatial understanding.

In [ ]:
def get_per_class_accuracy(model, loader):
    model.eval()
    class_correct = [0] * 10
    class_total = [0] * 10
    with torch.inference_mode():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            preds = model(X).argmax(1)
            for i in range(len(y)):
                label = y[i].item()
                class_total[label] += 1
                if preds[i] == label:
                    class_correct[label] += 1
    return [class_correct[i] / class_total[i] * 100 for i in range(10)]

mlp_per_class = get_per_class_accuracy(mlp_model, test_loader)
cnn_per_class = get_per_class_accuracy(cnn_model, test_loader)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(10)
width = 0.35

bars1 = ax.bar(x - width/2, mlp_per_class, width, label='MLP', color='#EF4444', alpha=0.8)
bars2 = ax.bar(x + width/2, cnn_per_class, width, label='CNN', color='#4ADE80', alpha=0.8)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Per-Class Accuracy: MLP vs CNN', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=30, ha='right')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(60, 100)

# Annotate improvements
for i in range(10):
    diff = cnn_per_class[i] - mlp_per_class[i]
    if abs(diff) > 1:
        ax.annotate(f'+{diff:.0f}%', xy=(x[i] + width/2, cnn_per_class[i]),
                    ha='center', va='bottom', fontsize=8, color='#4ADE80', fontweight='bold')

plt.tight_layout()
plt.show()

# Biggest improvements
diffs = [(class_names[i], cnn_per_class[i] - mlp_per_class[i]) for i in range(10)]
diffs.sort(key=lambda x: -x[1])
print("Biggest CNN improvements:")
for name, d in diffs[:3]:
    print(f"  {name}: +{d:.1f}%")
print("\nClasses with similar shapes (Shirt/T-shirt/Pullover) benefit most from spatial features.")

---
# Part 3: Visualize What the CNN Learned (~10 min)

One of the best things about CNNs: we can actually see what the network learned. First-layer filters are directly interpretable.

### 3.1 Learned Filters (Layer 1)

Each filter in the first conv layer operates directly on the input image. We can visualize them as 3×3 grayscale patterns — they should look like edge detectors.

In [ ]:
# Get the learned filters from conv1
filters = cnn_model.conv1.weight.data.cpu()
print(f"Conv1 filters shape: {filters.shape}  → (32 filters, 1 channel, 3×3)")

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
fig.suptitle('Learned Filters — Conv Layer 1 (32 filters)', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    f = filters[i, 0]  # ith filter, single channel
    im = ax.imshow(f, cmap='RdBu_r', vmin=-f.abs().max(), vmax=f.abs().max())
    ax.set_title(f'F{i}', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Red/blue = positive/negative weights.")
print("Notice: many look like edge detectors at different angles.")
print("The network discovered these on its own — nobody told it to look for edges!")

### 3.2 Feature Maps — What the CNN "Sees"

Passing a real image through the trained CNN and looking at the activations at each layer. Early layers detect simple patterns, deeper layers detect complex ones.

In [ ]:
# Get a sample image
sample, label = test_dataset[7]
x = sample.unsqueeze(0).to(device)

# Extract activations at each conv layer
cnn_model.eval()
with torch.inference_mode():
    # Layer 1
    a1 = F.relu(cnn_model.bn1(cnn_model.conv1(x)))
    p1 = cnn_model.pool(a1)
    # Layer 2
    a2 = F.relu(cnn_model.bn2(cnn_model.conv2(p1)))
    p2 = cnn_model.pool(a2)
    # Layer 3
    a3 = F.relu(cnn_model.bn3(cnn_model.conv3(p2)))
    p3 = cnn_model.pool(a3)

activations = [
    ('Conv1 — Edges (28×28, 32ch)', a1),
    ('Conv2 — Textures (14×14, 64ch)', a2),
    ('Conv3 — Parts (7×7, 128ch)', a3),
]

fig, axes = plt.subplots(3, 9, figsize=(16, 6))

for row, (name, act) in enumerate(activations):
    # Show original image in first column
    if row == 0:
        axes[row, 0].imshow(sample.squeeze().cpu(), cmap='gray')
        axes[row, 0].set_title(f'{class_names[label]}', fontsize=9, fontweight='bold')
    else:
        axes[row, 0].axis('off')
    axes[row, 0].set_ylabel(name, fontsize=9)

    # Show 8 feature maps
    act_np = act[0].cpu().numpy()
    for col in range(8):
        axes[row, col + 1].imshow(act_np[col], cmap='viridis')
        axes[row, col + 1].axis('off')
        if row == 0:
            axes[row, col + 1].set_title(f'Ch {col}', fontsize=8)

fig.suptitle('Feature Maps — What Each Layer Activates On', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Layer 1 (top): responds to edges and gradients — clearly visible patterns")
print("Layer 2 (mid): combines edges into textures and corners")
print("Layer 3 (bot): tiny 7×7 maps capturing high-level part information")
print("\nThis is the hierarchical feature learning that makes CNNs powerful.")

### 3.3 Sample Predictions

Finally, let's see the CNN's predictions on some test images with confidence scores.

In [ ]:
cnn_model.eval()
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))

indices = np.random.choice(len(test_dataset), 12, replace=False)

for idx, ax in zip(indices, axes.flat):
    img, label = test_dataset[idx]
    with torch.inference_mode():
        logits = cnn_model(img.unsqueeze(0).to(device))
        probs = F.softmax(logits, dim=1)
        conf, pred = probs.max(1)

    # Unnormalize for display
    display_img = img.squeeze() * 0.3530 + 0.2860
    ax.imshow(display_img.cpu(), cmap='gray')

    correct = pred.item() == label
    color = '#4ADE80' if correct else '#EF4444'
    ax.set_title(f'{class_names[pred.item()]}\n{conf.item():.0%}',
                 fontsize=9, color=color, fontweight='bold')
    ax.axis('off')

fig.suptitle('CNN Predictions (green = correct, red = wrong)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Model Saving

In [ ]:
torch.save(cnn_model.state_dict(), 'fashion_cnn_model.pth')
print("CNN model saved to fashion_cnn_model.pth")

---
# Recap

### 1. CNNs Preserve Spatial Structure
MLPs flatten images into vectors, destroying spatial info. CNNs operate on 2D grids with local filters.

### 2. The Conv Block Pattern
```python
nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)  # Extract features
nn.ReLU()                                             # Non-linearity
nn.MaxPool2d(2, 2)                                    # Downsample
```
Stack these, increase channels, then flatten into a classifier.

### 3. Same Training Loop
Forward → Loss → Zero Grad → Backward → Step. Didn't change from week 1 or 2.

### 4. What's Next (Week 4)
Transfer learning — instead of training from scratch, start from a model that already learned features on millions of images.

---
*Questions?*